# TrainLM: a simple from-scratch pretraining run

Open this notebook in Colab or Kaggle with a **single-device native TorchTPU runtime**. It starts a small GPT-2 model with random weights, learns a tokenizer from TinyStories, trains for a small token budget, and downloads the exported model and run report.

You only need to choose a token budget and sequence length. The data preparation, model defaults, package checks, TPU worker process, logging, and export are handled for you. This is an educational smoke run: it does not establish convergence, useful model quality, or TPU performance. The native TorchTPU training path is experimental.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-native-scratch-source-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch",
    "main",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
print(f"source_revision={source_revision}")

## 1. Choose a small run

For your first run, change only `TARGET_SUPERVISED_TOKENS` and, if needed, `SEQUENCE_LENGTH` below. The default uses 10,000 supervised next-token targets, a short exercise to verify the end-to-end path rather than train a useful model.

The notebook uses a pinned TinyStories revision so the data is reproducible. To use another Hugging Face dataset, set its dataset ID, immutable 40-character revision, split names, and text column in the same cell. Keep training and validation splits separate. The model size and optimizer are safe starter defaults managed by the helper; advanced runtime checks and worker arguments are kept out of the notebook.

In [ ]:
from pathlib import Path
import runpy
import tempfile

# Beginner settings
TARGET_SUPERVISED_TOKENS = 10_000
SEQUENCE_LENGTH = 128

# Optional data-source settings; keep the pinned defaults for a first run.
DATASET_ID = "roneneldan/TinyStories"
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
TRAIN_SPLIT = "train"
VALIDATION_SPLIT = "validation"
TEXT_FIELD = "text"

session_dir = Path(tempfile.mkdtemp(prefix="trainlm-pretraining-"))
output_dir = session_dir / "artifacts"
report_path = session_dir / "report.json"
print(f"source_revision={source_revision}")
print(f"run_artifacts={session_dir}")
print(f"target_supervised_tokens={TARGET_SUPERVISED_TOKENS}")

## 2. Run pretraining

This single call checks the installed package profile, prepares the tokenizer and packed data, and starts training in a separate process so TPU runtime state stays isolated from the notebook kernel. It prints a JSON report when the run finishes. If it stops early, the report and raw logs remain in `session_dir` for inspection.

In [ ]:
run_pretraining = runpy.run_path(
    str(repo_root / "examples/torchtpu_pretraining_quickstart.py")
)["run_pretraining"]
report = run_pretraining(
    repo_root=repo_root,
    output_dir=output_dir,
    report_path=report_path,
    source_revision=source_revision,
    target_supervised_tokens=TARGET_SUPERVISED_TOKENS,
    sequence_length=SEQUENCE_LENGTH,
    dataset_id=DATASET_ID,
    dataset_revision=DATASET_REVISION,
    train_split=TRAIN_SPLIT,
    validation_split=VALIDATION_SPLIT,
    text_field=TEXT_FIELD,
)

## 3. Download the result

The ZIP contains the Hugging Face model/tokenizer export and the machine-readable run report. Colab starts a download; Kaggle saves the ZIP under `/kaggle/working` for the notebook output panel. If preflight or training fails, run the next cell to download the report without a model export.

In [ ]:
from pathlib import Path
import shutil
import zipfile

export_dir = session_dir / "artifacts" / "baseline-export"
required = ("config.json", "tokenizer_config.json")
missing = [name for name in required if not (export_dir / name).is_file()]
weights_present = (
    any(path.is_file() and path.suffix in {".safetensors", ".bin"}
        for path in export_dir.rglob("*"))
    if export_dir.is_dir() else False
)
model_export_ready = not missing and weights_present
if not report_path.is_file():
    raise RuntimeError("The evidence report is missing.")

artifact_bundle = session_dir / "trainlm-scratch-artifact.zip"
with zipfile.ZipFile(artifact_bundle, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(report_path, arcname="report.json")
    if model_export_ready:
        for path in sorted(export_dir.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=Path("model") / path.relative_to(export_dir))
if model_export_ready:
    print("model_export=included")
else:
    print(f"model_export=not_included missing_files={missing} weights_present={weights_present}")

kaggle_output = Path("/kaggle/working")
if kaggle_output.is_dir():
    kaggle_artifact = kaggle_output / artifact_bundle.name
    shutil.copy2(artifact_bundle, kaggle_artifact)
    print(f"kaggle_artifact={kaggle_artifact}")
try:
    from google.colab import files
except ImportError:
    print(f"artifact_bundle={artifact_bundle}")
else:
    files.download(str(artifact_bundle))